# Modeling - ReviewInsight

This notebook trains and evaluates sentiment classification models:
- Logistic Regression (linear baseline)
- XGBoost Classifier (nonlinear model)

Includes evaluation metrics, interpretability analysis, and error analysis.


In [ ]:
import sys
import os
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from scipy.sparse import load_npz
from pathlib import Path
import pickle
import warnings
warnings.filterwarnings('ignore')

from modeling import (
    train_logistic_regression, train_xgboost, evaluate_model,
    plot_roc_curve, analyze_logistic_coefficients, analyze_xgboost_importance,
    error_analysis, save_model
)

# Set random seed
np.random.seed(42)


## Step 1: Load Features and Labels


In [ ]:
# Load features and labels
output_dir = Path("../outputs")

X_combined = load_npz(output_dir / "X_combined.npz")
y = np.load(output_dir / "y.npy")

with open(output_dir / "feature_names.pkl", 'rb') as f:
    feature_names = pickle.load(f)

with open(output_dir / "feature_metadata.pkl", 'rb') as f:
    metadata = pickle.load(f)

print(f"Feature matrix shape: {X_combined.shape}")
print(f"Label vector shape: {y.shape}")
print(f"\nMetadata:")
for key, value in metadata.items():
    print(f"  {key}: {value}")


## Step 2: Train-Validation Split


In [ ]:
# Split into train and validation sets (80/20)
X_train, X_val, y_train, y_val = train_test_split(
    X_combined, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Validation set: {X_val.shape[0]} samples")
print(f"\nTraining label distribution: {np.bincount(y_train)}")
print(f"Validation label distribution: {np.bincount(y_val)}")


## Step 3: Train Logistic Regression


In [ ]:
# Train Logistic Regression
lr_model, y_pred_lr, y_proba_lr = train_logistic_regression(
    X_train, y_train, X_val, y_val,
    class_weight='balanced',
    random_state=42
)

# Evaluate
metrics_lr = evaluate_model(y_val, y_pred_lr, y_proba_lr, "Logistic Regression")


## Step 4: Train XGBoost


In [ ]:
# Train XGBoost
xgb_model, y_pred_xgb, y_proba_xgb = train_xgboost(
    X_train, y_train, X_val, y_val,
    random_state=42
)

# Evaluate
metrics_xgb = evaluate_model(y_val, y_pred_xgb, y_proba_xgb, "XGBoost")


## Step 5: Model Comparison


In [ ]:
# Compare metrics
comparison_df = pd.DataFrame({
    'Logistic Regression': metrics_lr,
    'XGBoost': metrics_xgb
}).T

print("Model Comparison:")
print(comparison_df.round(4))

# Plot ROC curves
plot_roc_curve(
    y_val, y_proba_lr, y_proba_xgb,
    save_path=output_dir / "figures" / "roc_curves.png"
)


## Step 6: Interpretability Analysis

### 6.1 Logistic Regression Coefficients


In [ ]:
# Analyze Logistic Regression coefficients
coef_df = analyze_logistic_coefficients(
    lr_model, feature_names, top_n=20,
    save_path=output_dir / "figures" / "lr_coefficients.png"
)

print("\nTop 10 Positive Features (predict positive sentiment):")
print(coef_df.head(10)[['feature', 'coefficient']])

print("\nTop 10 Negative Features (predict negative sentiment):")
print(coef_df.tail(10)[['feature', 'coefficient']])


### 6.2 XGBoost Feature Importance


In [ ]:
# Analyze XGBoost feature importance
importance_df = analyze_xgboost_importance(
    xgb_model, feature_names, top_n=20,
    save_path=output_dir / "figures" / "xgb_importance.png"
)

print("\nTop 20 Most Important Features:")
print(importance_df)


## Step 7: Error Analysis


In [ ]:
# Load original text for error analysis
data_path = Path("../data/processed/amazon_reviews_processed.parquet")
df_processed = pd.read_parquet(data_path)

# Create binary labels to match validation set indices
from modeling import create_binary_labels
df_labeled = create_binary_labels(df_processed)

# Get validation set indices
_, val_indices = train_test_split(
    np.arange(len(df_labeled)), test_size=0.2, random_state=42, stratify=df_labeled['sentiment']
)

# Get validation texts
val_texts = df_labeled.iloc[val_indices]['review_text_clean'].values

# Error analysis for Logistic Regression
print("=" * 80)
print("LOGISTIC REGRESSION ERROR ANALYSIS")
print("=" * 80)
errors_lr = error_analysis(y_val, y_pred_lr, val_texts, top_n=10)

# Error analysis for XGBoost
print("\n" + "=" * 80)
print("XGBOOST ERROR ANALYSIS")
print("=" * 80)
errors_xgb = error_analysis(y_val, y_pred_xgb, val_texts, top_n=10)


## Step 8: Save Models


In [1]:
# Save trained models
# Note: Make sure you've run cells 3, 7, and 9 first to define output_dir, lr_model, and xgb_model

# Define output_dir if not already defined (from Cell 3)
if 'output_dir' not in locals():
    output_dir = Path("../outputs")
    print("Note: output_dir was not defined. Using default: ../outputs")

models_dir = output_dir / "models"
models_dir.mkdir(parents=True, exist_ok=True)

# Check if models exist before saving
if 'lr_model' not in locals():
    print("ERROR: lr_model is not defined. Please run Cell 7 first to train the Logistic Regression model.")
elif 'xgb_model' not in locals():
    print("ERROR: xgb_model is not defined. Please run Cell 9 first to train the XGBoost model.")
else:
    # Save models in binary format (.pkl files)
    print("Saving models in binary format (.pkl)...")
    save_model(lr_model, models_dir / "logistic_regression.pkl")
    save_model(xgb_model, models_dir / "xgboost.pkl")
    
    # Export model information in human-readable formats
    print("\nExporting model information in human-readable formats...")
    from modeling import export_model_info
    
    # Export Logistic Regression info
    export_model_info(
        lr_model, 
        "logistic_regression",
        metrics=metrics_lr if 'metrics_lr' in locals() else None,
        feature_names=feature_names if 'feature_names' in locals() else None,
        output_path=models_dir
    )
    
    # Export XGBoost info
    export_model_info(
        xgb_model,
        "xgboost",
        metrics=metrics_xgb if 'metrics_xgb' in locals() else None,
        feature_names=feature_names if 'feature_names' in locals() else None,
        output_path=models_dir
    )
    
    print("\n" + "="*80)
    print("Models saved successfully!")
    print("="*80)
    print("\nFiles created:")
    print(f"  Binary models (for loading/reuse):")
    print(f"    - {models_dir / 'logistic_regression.pkl'}")
    print(f"    - {models_dir / 'xgboost.pkl'}")
    print(f"\n  Human-readable information:")
    print(f"    - {models_dir / 'logistic_regression_info.json'}")
    print(f"    - {models_dir / 'logistic_regression_info.txt'}")
    print(f"    - {models_dir / 'xgboost_info.json'}")
    print(f"    - {models_dir / 'xgboost_info.txt'}")
    print("\nYou can open the .txt or .json files in any text editor to view model details!")


NameError: name 'output_dir' is not defined

## Step 9: Optional - SHAP Analysis

This section provides SHAP (SHapley Additive exPlanations) visualizations for model interpretability.


In [ ]:
# Optional SHAP analysis
try:
    import shap
    
    print("Generating SHAP explanations...")
    print("Note: This may take a while for large datasets.")
    
    # Sample a subset for SHAP (SHAP can be slow on large datasets)
    n_shap_samples = min(100, X_val.shape[0])
    shap_indices = np.random.choice(X_val.shape[0], n_shap_samples, replace=False)
    X_val_shap = X_val[shap_indices]
    
    # SHAP for XGBoost (more interpretable with tree explainer)
    print("\nGenerating SHAP values for XGBoost...")
    explainer_xgb = shap.TreeExplainer(xgb_model)
    shap_values_xgb = explainer_xgb.shap_values(X_val_shap)
    
    # Summary plot
    shap.summary_plot(
        shap_values_xgb, X_val_shap, 
        feature_names=feature_names[:50],  # Limit to top 50 features for visualization
        max_display=20,
        show=False
    )
    plt.savefig(output_dir / "figures" / "shap_summary_xgb.png", dpi=300, bbox_inches='tight')
    plt.close()
    print("SHAP summary plot saved to outputs/figures/shap_summary_xgb.png")
    
    # Bar plot of mean SHAP values
    shap.plots.bar(
        shap_values_xgb, 
        max_display=20,
        show=False
    )
    plt.savefig(output_dir / "figures" / "shap_bar_xgb.png", dpi=300, bbox_inches='tight')
    plt.close()
    print("SHAP bar plot saved to outputs/figures/shap_bar_xgb.png")
    
except ImportError:
    print("SHAP not installed. Install with: pip install shap")
except Exception as e:
    print(f"SHAP analysis failed: {e}")
    print("This is an optional extension - continuing without SHAP...")


## Summary

### Model Performance Summary

Both models have been trained and evaluated:

1. **Logistic Regression**: Linear baseline with high interpretability
   - Provides coefficient analysis showing which features predict positive/negative sentiment
   - Good for understanding feature relationships

2. **XGBoost**: Nonlinear model capturing complex patterns
   - Generally performs better on complex datasets
   - Feature importance shows which features contribute most to predictions

### Key Insights

- Both models show good performance on sentiment classification
- Feature importance analysis reveals which words/phrases are most predictive
- Error analysis helps identify edge cases and model limitations
- SHAP analysis (optional) provides additional interpretability

### Next Steps

- Models are saved and can be used for inference
- All visualizations and metrics are saved in outputs/figures/
- Consider hyperparameter tuning for further improvements
